In [2]:
from datetime import datetime
import time
from pyspark.sql import functions as F

GOLD_SCHEMA = "gold"
MAINTENANCE_LOG_TABLE = "table_maintenance_log"

DEFAULT_RETENTION_HOURS = 168  # 7 days

# Tables that usually need frequent optimization
OPTIMIZE_TABLES = [
    "gold.fact_orders",
]

# Tables that can be maintained less frequently
WEEKLY_MAINTENANCE_TABLES = [
    "gold.dim_customers",
    "gold.dim_products",
    "gold.dim_stores",
    "gold.dim_employees",
    "gold.dim_orders",
]

# Threshold to avoid unnecessary OPTIMIZE
MIN_FILE_COUNT_FOR_OPTIMIZE = 500

StatementMeta(, c1cc99b6-2727-4f7a-a1d9-78ebf1ea5eda, 4, Finished, Available, Finished, False)

In [3]:
GOLD_TABLES = [
    "gold.fact_orders",
    "gold.dim_customers",
    "gold.dim_products",
    "gold.dim_stores",
    "gold.dim_employees",
    "gold.dim_orders",
]

StatementMeta(, c1cc99b6-2727-4f7a-a1d9-78ebf1ea5eda, 5, Finished, Available, Finished, False)

In [4]:
def get_table_metrics(table_name: str) -> dict:
    """
    Get Delta table information before maintenance.
    """

    try:
        detail = spark.sql(
            f"DESCRIBE DETAIL {table_name}"
        ).collect()[0]

        return {
            "file_count": detail["numFiles"],
            "size_bytes": detail["sizeInBytes"]
        }

    except Exception as e:
        print(f"[DETAIL] {table_name} FAILED: {e}")

        return {
            "file_count": None,
            "size_bytes": None
        }


def optimize_table(table_name: str) -> tuple:

    start_time = time.time()

    try:
        metrics = get_table_metrics(table_name)

        file_count = metrics["file_count"]

        if file_count is not None and file_count < MIN_FILE_COUNT_FOR_OPTIMIZE:
            print(
                f"[OPTIMIZE] {table_name} skipped "
                f"(files={file_count})"
            )

            return "SKIPPED", 0

        print(f"[OPTIMIZE] {table_name} started")

        spark.sql(
            f"OPTIMIZE {table_name}"
        )

        duration = round(
            time.time() - start_time,
            2
        )

        print(
            f"[OPTIMIZE] {table_name} completed "
            f"({duration}s)"
        )

        return "SUCCESS", duration


    except Exception as e:

        duration = round(
            time.time() - start_time,
            2
        )

        print(
            f"[OPTIMIZE] {table_name} FAILED: {e}"
        )

        return "FAILED", duration



def vacuum_table(
    table_name: str,
    retention_hours: int = DEFAULT_RETENTION_HOURS
) -> tuple:

    start_time = time.time()

    try:

        print(
            f"[VACUUM] {table_name} started "
            f"(retention={retention_hours}h)"
        )

        spark.sql(
            f"""
            VACUUM {table_name}
            RETAIN {retention_hours} HOURS
            """
        )

        duration = round(
            time.time() - start_time,
            2
        )

        print(
            f"[VACUUM] {table_name} completed "
            f"({duration}s)"
        )

        return "SUCCESS", duration


    except Exception as e:

        duration = round(
            time.time() - start_time,
            2
        )

        print(
            f"[VACUUM] {table_name} FAILED: {e}"
        )

        return "FAILED", duration



def maintain_table(
    table_name: str,
    run_vacuum: bool = False,
    retention_hours: int = DEFAULT_RETENTION_HOURS
) -> dict:

    print("=" * 80)
    print(f"Maintaining: {table_name}")
    print("=" * 80)

    optimize_status, optimize_duration = optimize_table(
        table_name
    )

    if run_vacuum:
        vacuum_status, vacuum_duration = vacuum_table(
            table_name,
            retention_hours
        )
    else:
        vacuum_status = "SKIPPED"
        vacuum_duration = 0
    return {
        "table_name": table_name,
        "optimize_status": optimize_status,
        "vacuum_status": vacuum_status,
        "optimize_duration_seconds": optimize_duration,
        "vacuum_duration_seconds": vacuum_duration,
        "execution_timestamp": datetime.now(),
    }

StatementMeta(, c1cc99b6-2727-4f7a-a1d9-78ebf1ea5eda, 6, Finished, Available, Finished, False)

In [5]:

RUN_VACUUM = True
tables_to_process = (
    OPTIMIZE_TABLES
    + WEEKLY_MAINTENANCE_TABLES
)


maintenance_results = [
    maintain_table(
        table_name,
        run_vacuum=RUN_VACUUM,
        retention_hours=DEFAULT_RETENTION_HOURS
    )
    for table_name in tables_to_process
]

StatementMeta(, c1cc99b6-2727-4f7a-a1d9-78ebf1ea5eda, 7, Finished, Available, Finished, False)

Maintaining: gold.fact_orders
[OPTIMIZE] gold.fact_orders skipped (files=1)
[VACUUM] gold.fact_orders started (retention=168h)
[VACUUM] gold.fact_orders completed (45.06s)
Maintaining: gold.dim_customers
[OPTIMIZE] gold.dim_customers skipped (files=1)
[VACUUM] gold.dim_customers started (retention=168h)
[VACUUM] gold.dim_customers completed (11.43s)
Maintaining: gold.dim_products
[OPTIMIZE] gold.dim_products skipped (files=1)
[VACUUM] gold.dim_products started (retention=168h)
[VACUUM] gold.dim_products completed (10.76s)
Maintaining: gold.dim_stores
[OPTIMIZE] gold.dim_stores skipped (files=1)
[VACUUM] gold.dim_stores started (retention=168h)
[VACUUM] gold.dim_stores completed (7.85s)
Maintaining: gold.dim_employees
[OPTIMIZE] gold.dim_employees skipped (files=1)
[VACUUM] gold.dim_employees started (retention=168h)
[VACUUM] gold.dim_employees completed (8.27s)
Maintaining: gold.dim_orders
[OPTIMIZE] gold.dim_orders skipped (files=1)
[VACUUM] gold.dim_orders started (retention=168h)
[V

In [6]:
summary_df = spark.createDataFrame(
    maintenance_results
).select(
    "table_name",
    "optimize_status",
    "vacuum_status",
    "optimize_duration_seconds",
    "vacuum_duration_seconds",
    "execution_timestamp"
)


display(summary_df)


full_log_table = f"{GOLD_SCHEMA}.{MAINTENANCE_LOG_TABLE}"


(
    summary_df.write
    .format("delta")
    .mode("append")
    .option(
        "mergeSchema",
        "true"
    )
    .saveAsTable(full_log_table)
)


failed_count = sum(
    1
    for r in maintenance_results
    if r["optimize_status"] == "FAILED"
    or r["vacuum_status"] == "FAILED"
)


print(
    f"Maintenance complete: "
    f"{len(maintenance_results)} tables processed, "
    f"{failed_count} failure(s)"
)


if failed_count > 0:
    raise Exception(
        f"Maintenance failed for {failed_count} table(s)"
    )

StatementMeta(, c1cc99b6-2727-4f7a-a1d9-78ebf1ea5eda, 8, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 3311a561-59f3-42f8-aef9-4600af2262fe)

Maintenance complete: 6 tables processed, 0 failure(s)


In [1]:
%%sql
-- drop table gold.table_maintenance_log

StatementMeta(, c1cc99b6-2727-4f7a-a1d9-78ebf1ea5eda, 2, Finished, Available, Finished, False)

<Spark SQL result set with 0 rows and 0 fields>